In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

df = pd.read_csv("dados_sensores_vulcantech.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(by=["id_celula", "timestamp"]).reset_index(drop=True)

print(f"Dimensões do dataset original: {df.shape}")
df.head()

In [ ]:
sensor_cols = [
    "temperatura_fuso_c",
    "vibracao_triaxial_rms_mms",
    "torque_fuso_nm",
    "vazao_refrigerante_lmin",
    "corrente_estator_amp",
    "pressao_hidraulica_bar",
    "nivel_ruido_db"
]

taxa_nans = (
    df.groupby(["id_celula", "liga_metalica"])[sensor_cols]
    .apply(lambda g: (g.isna().mean() * 100))
    .round(2)
)

print("--- Taxa de Ausência de Dados (NaNs em %) por Célula e Liga ---")
display(taxa_nans)

In [ ]:
df_sanitizado = df.copy()

# Limites físicos estritos de viabilidade (sensores desconectados / estouro de escala)
df_sanitizado.loc[df_sanitizado["temperatura_fuso_c"] >= 900.0, "temperatura_fuso_c"] = np.nan
df_sanitizado.loc[df_sanitizado["vibracao_triaxial_rms_mms"] < 0.0, "vibracao_triaxial_rms_mms"] = np.nan
df_sanitizado.loc[df_sanitizado["pressao_hidraulica_bar"] < 0.0, "pressao_hidraulica_bar"] = np.nan
df_sanitizado.loc[df_sanitizado["vazao_refrigerante_lmin"] < 0.0, "vazao_refrigerante_lmin"] = np.nan

# Diagnóstico pós-conversão de ruídos em NaNs
nans_pos_ruido = df_sanitizado[sensor_cols].isna().sum()
print("Volume total de lacunas a serem interpoladas (originais + ruídos saneados):")
print(nans_pos_ruido)

In [ ]:
# Ordenar estritamente por célula e timestamp 
df_sanitizado = df_sanitizado.sort_values(by=["id_celula", "timestamp"]).reset_index(drop=True)

# Interpolação temporal agrupada por id_celula
df_interpolado = df_sanitizado.copy()

df_interpolado[sensor_cols] = (
    df_sanitizado.groupby("id_celula")[sensor_cols]
    .transform(lambda grupo: grupo.interpolate(method="linear").bfill().ffill())
)

print(f"NaNs remanescentes após interpolação condicional: {df_interpolado[sensor_cols].isna().sum().sum()}")

In [ ]:
df_features = df_interpolado.copy()

# 1. Feature Derivada: Índice de Esforço Mecânico (IEM)
df_features["indice_esforco_mecanico"] = (
    df_features["torque_fuso_nm"] * df_features["vibracao_triaxial_rms_mms"]
) / (df_features["vazao_refrigerante_lmin"] + 1.0)

# 2. Sensores críticos para monitoramento de degradação temporal
sensores_dinamicos = ["temperatura_fuso_c", "vibracao_triaxial_rms_mms", "torque_fuso_nm", "indice_esforco_mecanico"]

# 3. Rolling Features por célula (amostragem a cada 5 min: 6 passos = 30min; 24 passos = 2h)
janelas = {"30m": 6, "2h": 24}

for rotulo, w in janelas.items():
    for col in sensores_dinamicos:
        df_features[f"{col}_roll_mean_{rotulo}"] = (
            df_features.groupby("id_celula")[col]
            .transform(lambda x: x.rolling(window=w, min_periods=1).mean())
        )
        df_features[f"{col}_roll_std_{rotulo}"] = (
            df_features.groupby("id_celula")[col]
            .transform(lambda x: x.rolling(window=w, min_periods=1).std().fillna(0.0))
        )

print("Novas colunas geradas:")
print([col for col in df_features.columns if "roll" in col or "esforco" in col])

In [ ]:
# Normalização condicional: Razão em relação à média histórica da respectiva liga metálica
for col in sensor_cols + ["indice_esforco_mecanico"]:
    media_liga = df_features.groupby("liga_metalica")[col].transform("mean")
    df_features[f"{col}_ratio_liga"] = df_features[col] / media_liga

# Validação do baseline: média das variáveis normalizadas por liga deve ser ~1.0
validacao_norm = df_features.groupby("liga_metalica")[[f"{c}_ratio_liga" for c in ["torque_fuso_nm", "temperatura_fuso_c"]]].mean()
print("--- Validação da Normalização por Liga (Médias devem ser iguais a 1.0) ---")
display(validacao_norm)

In [ ]:
# Visualização do impacto da normalização na eliminação de viés de material
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_features, x="liga_metalica", y="torque_fuso_nm", ax=axes[0])
axes[0].set_title("Torque Bruto por Liga (Diferenças Nominais Evidentes)")
axes[0].set_ylabel("Torque (Nm)")

sns.boxplot(data=df_features, x="liga_metalica", y="torque_fuso_nm_ratio_liga", ax=axes[1])
axes[1].set_title("Torque Normalizado por Liga (Baselines Equalizados)")
axes[1].set_ylabel("Razão s/ Média da Liga")

plt.tight_layout()
plt.show()

# Exportar base tratada para uso na Etapa 2 (Modelagem Preditiva & OEE)
df_features.to_csv("dados_sensores_tratados.csv", index=False)
print("Base tratada e enriquecida exportada com sucesso: 'dados_sensores_tratados.csv'")